In [1]:
import torch
from torch import nn
from torch.nn import Module
from torch.utils.data import Dataset, random_split, DataLoader

import polars as pl

from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics import roc_auc_score, f1_score, confusion_matrix

In [2]:
TARGET_MAP = {"positive": 1, "negative": 0}

def filter_dataset(dataset):
    dataset = dataset.select(pl.col(["airline_sentiment", "text"]))
    dataset = dataset.filter(pl.col("airline_sentiment") != "neutral")

    dataset = dataset.with_columns(
        pl.col("airline_sentiment").replace_strict(TARGET_MAP)
    )
    return dataset
    
dataset = pl.read_csv("../datasets/AirlineTweets.csv")
dataset = filter_dataset(dataset)

In [3]:
train, test = train_test_split(dataset, test_size = 0.1, shuffle = True)

xtrain, ytrain = train["text"], train["airline_sentiment"]
xtest, ytest = test["text"], test["airline_sentiment"]

In [4]:
vectorizer = TfidfVectorizer(max_features = 2000)
X_train = vectorizer.fit_transform(xtrain).toarray()
X_test = vectorizer.transform(xtest).toarray()

In [5]:
class TweetDataset(Dataset):
    def __init__(self, X, y):
        self.X = torch.tensor(X, dtype=torch.float32)
        self.y = torch.tensor(y.to_numpy(), dtype=torch.float32)  # shape: (N,)
    
    def __len__(self):
        return len(self.X)

    def __getitem__(self, idx):
        return self.X[idx], self.y[idx]

train_loader = DataLoader(TweetDataset(X_train, ytrain), 64)
test_loader = DataLoader(TweetDataset(X_test, ytest), 64)

In [6]:
class ClassificationModel(nn.Module):
    def __init__(self):
        super().__init__()

        self.linear = nn.Sequential(
            nn.Linear(2000, 1)
        )

    def forward(self, x):
        return self.linear(x)

In [7]:
model = ClassificationModel()
loss = nn.BCEWithLogitsLoss()
optim = torch.optim.Adam(model.parameters(), lr = 0.0025)

In [8]:
losses = []

model.train()
for epoch in range(50):
    epoch_loss = 0
    for (x,y) in train_loader:
        loss.zero_grad()

        output = model(x)
        loss_ = loss(output, y.unsqueeze(-1))

        loss_.backward()
        optim.step()

        epoch_loss += loss_.item()
    
    epoch_loss /= len(train_loader)
    
    print(f"Epoch: {epoch}  Loss: {epoch_loss}")


Epoch: 0  Loss: 0.4933088952961144
Epoch: 1  Loss: 0.3657681453355982
Epoch: 2  Loss: 0.2668488531893382
Epoch: 3  Loss: 0.20283289596529827
Epoch: 4  Loss: 0.20473937283447183
Epoch: 5  Loss: 0.17211680892466402
Epoch: 6  Loss: 0.15796410006419556
Epoch: 7  Loss: 0.16431425257437388
Epoch: 8  Loss: 0.1520659083649059
Epoch: 9  Loss: 0.1343770796558791
Epoch: 10  Loss: 0.13811159906394643
Epoch: 11  Loss: 0.1422189172825839
Epoch: 12  Loss: 0.1278519875104282
Epoch: 13  Loss: 0.1227002814726367
Epoch: 14  Loss: 0.13096198735171585
Epoch: 15  Loss: 0.12801892984875599
Epoch: 16  Loss: 0.11468399949286111
Epoch: 17  Loss: 0.11565732705961067
Epoch: 18  Loss: 0.12384271196181103
Epoch: 19  Loss: 0.11465680694630358
Epoch: 20  Loss: 0.1065334858793474
Epoch: 21  Loss: 0.11408451870785515
Epoch: 22  Loss: 0.11488588296362282
Epoch: 23  Loss: 0.10302590364747412
Epoch: 24  Loss: 0.10300128765267097
Epoch: 25  Loss: 0.11308927702691772
Epoch: 26  Loss: 0.10688150046818941
Epoch: 27  Loss: 0.0

In [21]:
true_labels = []
outputs = []

with torch.no_grad():
    test_loss = 0
    for (x,y) in test_loader:
        output = model(x)
        loss_ = loss(output, y.unsqueeze(-1))
        test_loss += loss_.item()

        true_labels.extend(y)
        outputs.extend(y)

    test_loss /= len(test_loader)

print(test_loss)



0.8863264714416704


In [22]:
auc = roc_auc_score(true_labels, outputs)
f1score = f1_score(true_labels, outputs)

confusion_matrix(true_labels, outputs)

array([[920,   0],
       [  0, 235]])

In [23]:
w = model.state_dict()["linear.0.weight"]
word_index_map = vectorizer.vocabulary_

In [24]:
threshold = 2
word_weight_tuples = []

for word, index in word_index_map.items():
    weight = w[0][index]

    if weight > threshold:
        word_weight_tuples.append((word, weight))
        
word_weight_tuples.sort(key = lambda x: -x[1])

In [25]:
print("Most positive words")
print(*word_weight_tuples[:10], sep = "\n")

Most positive words
('exactly', tensor(25.3517))
('attention', tensor(25.1198))
('type', tensor(24.9619))
('offers', tensor(24.7370))
('complain', tensor(24.5210))
('impressed', tensor(24.4187))
('worries', tensor(24.2827))
('coat', tensor(24.2233))
('uk', tensor(24.2046))
('decent', tensor(24.1908))


In [26]:
print("Most negative words")
print(*word_weight_tuples[-10:], sep = "\n")

Most negative words
('issue', tensor(2.2706))
('on', tensor(2.2618))
('connecting', tensor(2.1995))
('course', tensor(2.1830))
('return', tensor(2.1683))
('tomorrow', tensor(2.1240))
('prices', tensor(2.0898))
('walk', tensor(2.0861))
('names', tensor(2.0580))
('area', tensor(2.0395))
